# 21 — Confidence Interval using statsmodels / 使用statsmodels的置信区间

**Chapter 21 — File 3 of 4**

## Summary / 摘要

The statsmodels library provides the proportion_confint() function, which automatically computes confidence intervals for proportions. This function handles multiple methods (normal, agresti_coull, binom_test, wilson, etc.) and abstracts away manual calculations. Using proportion_confint(88, 100, 0.05) with 88 successes out of 100 observations and α=0.05 significance level yields a 95% confidence interval. This approach is faster and more robust than manual calculation, especially for edge cases and small samples.

statsmodels库提供了proportion_confint()函数，它自动计算比例的置信区间。此函数处理多种方法（正态、agresti_coull、binom_test、wilson等）并抽象出手动计算。使用proportion_confint(88, 100, 0.05)，其中有88次成功，共100个观察，α=0.05显著性水平，产生95%置信区间。此方法比手动计算更快、更稳健，特别是对于边缘情况和小样本。

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import required libraries
# 导入所需库
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint
from scipy import stats

# Set random seed for reproducibility
# 设置随机种子以保证可重复性
np.random.seed(42)

## Step 2 — Define Sample Data / 定义样本数据

In [ ]:
# Sample data parameters
# 样本数据参数
n_observations = 100  # Total observations / 总观察数
n_successes = 88  # Number of successes / 成功次数
alpha = 0.05  # Significance level (for 95% CI) / 显著性水平（用于95% CI）

# Calculate sample proportion
# 计算样本比例
p_sample = n_successes / n_observations

# Display parameters
# 显示参数
print(f"Sample data:")
print(f"  Total observations: {n_observations}")
print(f"  Successes: {n_successes}")
print(f"  Sample proportion: {p_sample:.4f}")
print(f"  Significance level (α): {alpha}")
print(f"  Confidence level: {(1-alpha)*100:.0f}%")

## Step 3 — Calculate CI using statsmodels Default Method / 使用statsmodels默认方法计算置信区间

In [ ]:
# Calculate confidence interval using default method (normal approximation)
# 使用默认方法（正态近似）计算置信区间
ci_lower, ci_upper = proportion_confint(n_successes, n_observations, alpha=alpha, method='normal')

# Display confidence interval
# 显示置信区间
print(f"\n95% Confidence Interval (Normal method):")
print(f"  Lower bound: {ci_lower:.6f}")
print(f"  Upper bound: {ci_upper:.6f}")
print(f"  Interval: [{ci_lower:.6f}, {ci_upper:.6f}]")
print(f"  Or as percentages: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"  Width: {ci_upper - ci_lower:.6f}")
print(f"  Margin of error: {(ci_upper - ci_lower)/2:.6f}")

## Step 4 — Compare Different Methods / 比较不同的方法

In [ ]:
# Calculate CI using different methods
# 使用不同方法计算置信区间
methods = ['normal', 'agresti_coull', 'binom_test', 'wilson', 'beta', 'bayes']
results = {}

for method in methods:
    try:
        lower, upper = proportion_confint(n_successes, n_observations, alpha=alpha, method=method)
        results[method] = {'lower': lower, 'upper': upper, 'width': upper - lower}
    except:
        results[method] = {'lower': np.nan, 'upper': np.nan, 'width': np.nan}

# Display comparison table
# 显示比较表
print(f"\nComparison of Different CI Methods:")
print(f"{'Method':20} | {'Lower':10} | {'Upper':10} | {'Width':10}")
print("-" * 55)
for method in methods:
    if not np.isnan(results[method]['lower']):
        print(f"{method:20} | {results[method]['lower']:10.6f} | {results[method]['upper']:10.6f} | {results[method]['width']:10.6f}")

## Step 5 — Visualize Confidence Intervals / 可视化置信区间

In [ ]:
# Create visualization of different methods
# 创建不同方法的可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: All methods comparison
# 图1: 所有方法的比较
valid_methods = [m for m in methods if not np.isnan(results[m]['lower'])]
y_positions = np.arange(len(valid_methods))

lowers = [results[m]['lower'] for m in valid_methods]
uppers = [results[m]['upper'] for m in valid_methods]
centers = [(l + u) / 2 for l, u in zip(lowers, uppers)]
errors = [(u - l) / 2 for l, u in zip(lowers, uppers)]

axes[0, 0].errorbar(centers, y_positions, xerr=errors, fmt='o', markersize=8,
                    capsize=10, capthick=2, linewidth=2, color='blue')
axes[0, 0].axvline(p_sample, color='red', linestyle='--', linewidth=2, label='Sample proportion')
axes[0, 0].set_yticks(y_positions)
axes[0, 0].set_yticklabels(valid_methods)
axes[0, 0].set_xlabel('Proportion')
axes[0, 0].set_title('95% CI for Different Methods')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Plot 2: CI widths comparison
# 图2: 置信区间宽度比较
widths = [results[m]['width'] for m in valid_methods]
axes[0, 1].barh(valid_methods, widths, color='orange', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Interval Width')
axes[0, 1].set_title('Confidence Interval Width Comparison')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Plot 3: Sampling distribution with CI bounds
# 图3: 采样分布与置信区间边界
x = np.linspace(0.7, 1.0, 500)
y = stats.norm.pdf(x, p_sample, np.sqrt(p_sample * (1 - p_sample) / n_observations))

axes[1, 0].plot(x, y, 'b-', linewidth=2, label='Sampling distribution')
axes[1, 0].axvline(p_sample, color='black', linestyle='-', linewidth=2, label='Sample p')
axes[1, 0].axvline(ci_lower, color='green', linestyle='--', linewidth=2, label='CI bounds')
axes[1, 0].axvline(ci_upper, color='green', linestyle='--', linewidth=2)

x_fill = x[(x >= ci_lower) & (x <= ci_upper)]
y_fill = stats.norm.pdf(x_fill, p_sample, np.sqrt(p_sample * (1 - p_sample) / n_observations))
axes[1, 0].fill_between(x_fill, y_fill, alpha=0.3, color='green')

axes[1, 0].set_xlabel('Proportion')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Sampling Distribution with 95% CI')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Data summary
# 图4: 数据摘要
axes[1, 1].axis('off')
summary_text = f"""
Sample Data Summary:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Total observations: {n_observations}
Successes: {n_successes}
Failures: {n_observations - n_successes}

Sample proportion (p̂): {p_sample:.6f}
Significance level (α): {alpha}
Confidence level: {(1-alpha)*100:.0f}%

95% Confidence Interval (Normal):
  [{ci_lower:.6f}, {ci_upper:.6f}]
  [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]

Margin of error: ±{(ci_upper - ci_lower)/2:.6f}
Interval width: {ci_upper - ci_lower:.6f}
"""
axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
               verticalalignment='center')

plt.tight_layout()
plt.show()

In [ ]:
## Learning Notes / 学习笔记

- **Statistical Concept**: Different methods for computing confidence intervals exist because each makes different assumptions or trade-offs. The normal approximation is fast and adequate for large samples (n > 30). The Wilson method performs well across sample sizes and doesn't produce intervals outside [0,1]. Agresti-Coull adds 2 successes and 2 failures for improved small-sample properties.
  
  **统计概念**: 存在计算置信区间的不同方法，因为每种方法都做出了不同的假设或权衡。正态近似对于大样本（n > 30）是快速而充分的。Wilson方法在各种样本大小上表现良好，不会产生超出[0,1]范围的区间。Agresti-Coull添加2次成功和2次失败，以改进小样本性质。

- **ML Application**: In A/B testing and online experiments, selecting the right CI method impacts inference. The Wilson and Agresti-Coull methods are more conservative and better for small-sample scenarios common in early-stage experiments. Libraries like statsmodels handle edge cases automatically, reducing implementation errors in production systems where valid CIs are critical for business decisions.
  
  **ML应用**: 在A/B测试和在线实验中，选择正确的置信区间方法影响推断。Wilson和Agresti-Coull方法更保守，更适合早期实验中常见的小样本情景。statsmodels等库自动处理边缘情况，减少了生产系统中的实施错误，其中有效的置信区间对业务决策至关重要。

➡️ **Next**: `04_bootstrap_confidence_interval.ipynb`

## Complete Code / 完整代码一览

In [ ]:
from statsmodels.stats.proportion import proportion_confint
import numpy as np
import matplotlib.pyplot as plt

n_observations = 100
n_successes = 88
alpha = 0.05

p_sample = n_successes / n_observations
print(f"Sample proportion: {p_sample:.4f}")

# Using default method (normal)
ci_lower, ci_upper = proportion_confint(n_successes, n_observations, alpha=alpha, method='normal')
print(f"95% CI (normal): [{ci_lower:.6f}, {ci_upper:.6f}]")

# Using Wilson method (recommended)
ci_lower_w, ci_upper_w = proportion_confint(n_successes, n_observations, alpha=alpha, method='wilson')
print(f"95% CI (wilson): [{ci_lower_w:.6f}, {ci_upper_w:.6f}]")